In [ ]:
import numpy as np
import ifcopenshell
import ifcopenshell.api.root
import ifcopenshell.api.geometry

# assume model, body context and project already exist
project = model.by_type("IfcProject")[0]

# create the spatial structure if not already created
site = model.create_entity("IfcSite", Name="Site")
building = model.create_entity("IfcBuilding", Name="Building")
storey = model.create_entity("IfcBuildingStorey", Name="Level 1", Elevation=0.0)

model.create_entity(
    "IfcRelAggregates",
    RelatingObject=project,
    RelatedObjects=[site]
)
model.create_entity(
    "IfcRelAggregates",
    RelatingObject=site,
    RelatedObjects=[building]
)
model.create_entity(
    "IfcRelAggregates",
    RelatingObject=building,
    RelatedObjects=[storey]
)

# create the room space
space = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSpace",
    Name="Room 1",
    LongName="Office 101"
)

# place it in the model
matrix = np.eye(4)
matrix[:, 3][0:3] = (2.5, 2.1, 0.0)   # location of the room space
ifcopenshell.api.geometry.edit_object_placement(model, product=space,
    matrix=matrix, is_si=True)

# optional: give the space a simple rectangular geometry
profile = model.create_entity("IfcRectangleProfileDef",
    ProfileName="Room 1 footprint",
    ProfileType="AREA",
    XDim=5.4,
    YDim=4.2
)
space_rep = ifcopenshell.api.geometry.add_profile_representation(
    model,
    context=body,
    profile=profile,
    depth=3.0
)
ifcopenshell.api.geometry.assign_representation(
    model,
    product=space,
    representation=space_rep
)

# attach the space to the storey
model.create_entity(
    "IfcRelContainedInSpatialStructure",
    RelatingStructure=storey,
    RelatedElements=[space]
)